In [1]:
import os
import random
import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.utils.data import random_split

from torchvision import transforms

from sklearn.metrics import accuracy_score

from tqdm import tqdm

In [9]:
DATA_DIR = "data/Processed_Train/images"
CSV_PATH = "data/Processed_Train/processed_annotations.csv"

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

In [10]:
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-3 # 0.001
TRAIN_RATIO = 0.8
RANDOM_SEED = 42

In [11]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(DEVICE)

cuda


In [12]:
train_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(15),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

])

In [13]:
valid_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

])

In [14]:
class SnakeDataset(Dataset):

    def __init__(self, csv_file, image_dir, transform=None):

        self.annotations = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):

        row = self.annotations.iloc[index]

        image_path = os.path.join(
            self.image_dir,
            row["filename"]
        )

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = int(row["Label"])

        return image, label

In [15]:
dataset = SnakeDataset(
    csv_file=CSV_PATH,
    image_dir=DATA_DIR,
    transform=train_transform
)

print("Total Images:", len(dataset))

Total Images: 6109


In [16]:
train_size = int(TRAIN_RATIO * len(dataset))
valid_size = len(dataset) - train_size

train_dataset, valid_dataset = random_split(
    dataset,
    [train_size, valid_size],
    generator=torch.Generator().manual_seed(RANDOM_SEED)
)

In [17]:
train_dataset.dataset.transform = train_transform
valid_dataset.dataset.transform = valid_transform

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)